# Phase 1: Random Frequency Masking Visualization

This notebook provides comprehensive visualization tools for analyzing Phase 1 Random Frequency Masking augmentation strategy.

## Overview

Random Frequency Masking is a data augmentation technique inspired by SpecAugment (Park et al., 2019). It randomly masks frequency bands in spectrograms to improve model robustness by forcing it to learn from partial frequency information.

This notebook visualizes:
- Before/after comparison of masked spectrograms
- Statistical distribution of masked positions
- Parameter sensitivity analysis
- Impact on different spectrogram patterns

## 1. Setup

Import necessary libraries and configure paths for accessing the source code.

In [ ]:
import sys
import os
from pathlib import Path

# Add src directory to path for imports
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

print(f"Project root: {project_root}")
print(f"Source path: {src_path}")
print("Setup complete!")

## 2. Configuration

Import the RandomFrequencyMasker and configure masking parameters.

In [ ]:
from datamodules.components.frequency_masker import RandomFrequencyMasker

# Initialize masker with default parameters
masker = RandomFrequencyMasker(
    n_bands=2,           # Number of frequency bands to mask
    mask_prob=1.0,       # Always apply masking for visualization
    max_band_width=16,   # Maximum width of each masked band
    n_mels=128           # Total number of mel frequency bins
)

print("RandomFrequencyMasker initialized successfully!")
print(f"Configuration: {masker}")
print(f"\nParameters:")
print(f"  - n_bands: {masker.n_bands}")
print(f"  - mask_prob: {masker.mask_prob}")
print(f"  - max_band_width: {masker.max_band_width}")
print(f"  - n_mels: {masker.n_mels}")

## 3. Test Data Generation

Create synthetic spectrograms with known patterns to test the masking behavior.

In [ ]:
def create_synthetic_spectrogram(pattern='uniform', shape=(100, 128)):
    """
    Create synthetic spectrogram with different patterns.
    
    Args:
        pattern: Type of pattern to generate
            - 'uniform': Uniform random noise
            - 'horizontal_bands': Horizontal frequency bands
            - 'vertical_lines': Vertical time lines (calls)
            - 'diagonal': Diagonal pattern (frequency modulation)
            - 'gaussian_blob': Gaussian blob (localized call)
        shape: Tuple of (time, freq) dimensions
    
    Returns:
        Synthetic spectrogram with shape (time, freq)
    """
    time_steps, n_freq = shape
    
    if pattern == 'uniform':
        # Uniform random noise
        spec = np.random.randn(*shape) * 0.5 + 0.5
        
    elif pattern == 'horizontal_bands':
        # Horizontal frequency bands (different species occupy different frequencies)
        spec = np.zeros(shape)
        # Low frequency band
        spec[:, 20:40] = np.random.randn(time_steps, 20) * 0.3 + 1.0
        # Mid frequency band
        spec[:, 60:80] = np.random.randn(time_steps, 20) * 0.3 + 1.2
        # High frequency band
        spec[:, 100:120] = np.random.randn(time_steps, 20) * 0.3 + 0.8
        
    elif pattern == 'vertical_lines':
        # Vertical time lines (discrete calls)
        spec = np.random.randn(*shape) * 0.1 + 0.2
        for t in [20, 40, 60, 80]:
            spec[t-2:t+2, 40:90] = np.random.randn(4, 50) * 0.2 + 1.5
            
    elif pattern == 'diagonal':
        # Diagonal pattern (frequency modulation)
        spec = np.random.randn(*shape) * 0.1 + 0.2
        for t in range(time_steps):
            freq_center = int(30 + (t / time_steps) * 70)
            spec[t, max(0, freq_center-5):min(n_freq, freq_center+5)] = 1.5
            
    elif pattern == 'gaussian_blob':
        # Gaussian blob (localized call)
        spec = np.random.randn(*shape) * 0.1 + 0.2
        t_center, f_center = time_steps // 2, n_freq // 2
        for t in range(time_steps):
            for f in range(n_freq):
                dist = np.sqrt((t - t_center)**2 / 100 + (f - f_center)**2 / 400)
                spec[t, f] += 1.5 * np.exp(-dist)
    
    else:
        raise ValueError(f"Unknown pattern: {pattern}")
    
    # Clip to reasonable range
    spec = np.clip(spec, 0, 2)
    
    return spec

# Generate test spectrograms
print("Generating synthetic spectrograms...")
test_patterns = ['uniform', 'horizontal_bands', 'vertical_lines', 'diagonal', 'gaussian_blob']
test_specs = {pattern: create_synthetic_spectrogram(pattern) for pattern in test_patterns}
print(f"Generated {len(test_specs)} test spectrograms")
print(f"Spectrogram shape: {test_specs['uniform'].shape}")

## 4. Visualization Functions

### Function 1: Before/After Comparison

Visualizes the original and masked spectrograms side-by-side with red overlays highlighting masked regions.

In [ ]:
def visualize_before_after(original, masked, title="Frequency Masking Comparison"):
    """
    Visualize original and masked spectrograms side-by-side.
    
    Args:
        original: Original spectrogram with shape (time, freq)
        masked: Masked spectrogram with shape (time, freq)
        title: Title for the figure
    
    Note:
        - Uses viridis colormap for spectrograms
        - Adds red overlay to highlight masked regions
        - Shows difference map to visualize masking impact
    """
    # Validate inputs
    if original.shape != masked.shape:
        print(f"Error: Shape mismatch - original {original.shape} vs masked {masked.shape}")
        return
    
    # Compute difference (masked regions)
    difference = np.abs(original - masked)
    mask_binary = (difference > 0.01).astype(float)  # Binary mask of affected regions
    
    # Create figure with 3 subplots
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Left: Original spectrogram
    im1 = axes[0].imshow(original.T, aspect='auto', origin='lower', cmap='viridis',
                         extent=[0, original.shape[0], 0, original.shape[1]])
    axes[0].set_xlabel('Time Frame', fontsize=12)
    axes[0].set_ylabel('Frequency Bin', fontsize=12)
    axes[0].set_title('Original Spectrogram', fontsize=13, fontweight='bold')
    plt.colorbar(im1, ax=axes[0], label='Magnitude')
    
    # Middle: Masked spectrogram with red overlay
    im2 = axes[1].imshow(masked.T, aspect='auto', origin='lower', cmap='viridis',
                         extent=[0, masked.shape[0], 0, masked.shape[1]])
    # Overlay red mask on masked regions
    axes[1].imshow(mask_binary.T, aspect='auto', origin='lower', cmap='Reds',
                   alpha=0.3, extent=[0, masked.shape[0], 0, masked.shape[1]])
    axes[1].set_xlabel('Time Frame', fontsize=12)
    axes[1].set_ylabel('Frequency Bin', fontsize=12)
    axes[1].set_title('Masked Spectrogram (Red = Masked)', fontsize=13, fontweight='bold')
    plt.colorbar(im2, ax=axes[1], label='Magnitude')
    
    # Right: Difference map
    im3 = axes[2].imshow(difference.T, aspect='auto', origin='lower', cmap='hot',
                         extent=[0, difference.shape[0], 0, difference.shape[1]])
    axes[2].set_xlabel('Time Frame', fontsize=12)
    axes[2].set_ylabel('Frequency Bin', fontsize=12)
    axes[2].set_title('Difference (Masked Regions)', fontsize=13, fontweight='bold')
    plt.colorbar(im3, ax=axes[2], label='Absolute Difference')
    
    # Overall title
    fig.suptitle(title, fontsize=15, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    masked_fraction = np.sum(mask_binary) / mask_binary.size
    print(f"Masked fraction: {masked_fraction:.2%}")
    print(f"Number of masked bins: {np.sum(mask_binary):.0f} / {mask_binary.size}")

print("Function 1 (visualize_before_after) defined successfully!")

### Function 2: Masking Statistics

Analyzes the statistical distribution of masked positions across multiple samples.

In [ ]:
def visualize_masking_statistics(masker, n_samples=100):
    """
    Visualize statistical distribution of masked positions.
    
    Args:
        masker: RandomFrequencyMasker instance
        n_samples: Number of samples to generate for statistics
    
    Note:
        - Generates n_samples masked spectrograms
        - Computes frequency-wise masking probability
        - Shows histogram of masked band positions and widths
        - Validates uniform distribution of random masking
    """
    print(f"Generating {n_samples} samples for statistical analysis...")
    
    # Create test spectrogram
    test_spec = np.ones((100, masker.n_mels))
    
    # Collect masking statistics
    mask_counts = np.zeros(masker.n_mels)  # Count how often each frequency is masked
    band_starts = []  # Starting positions of masked bands
    band_widths = []  # Widths of masked bands
    
    for i in range(n_samples):
        if (i + 1) % 20 == 0:
            print(f"  Processing sample {i + 1}/{n_samples}...")
        
        masked = masker(test_spec)
        
        # Identify masked regions
        mask_binary = (masked[0, :] == 0).astype(int)
        mask_counts += mask_binary
        
        # Find band boundaries
        diff = np.diff(np.concatenate([[0], mask_binary, [0]]))
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]
        
        for start, end in zip(starts, ends):
            band_starts.append(start)
            band_widths.append(end - start)
    
    print(f"Analysis complete!\n")
    
    # Create figure with 3 subplots
    fig = plt.figure(figsize=(18, 5))
    gs = GridSpec(1, 3, figure=fig)
    
    # Left: Frequency-wise masking probability
    ax1 = fig.add_subplot(gs[0, 0])
    mask_prob = mask_counts / n_samples
    ax1.plot(mask_prob, linewidth=2, color='blue')
    ax1.fill_between(range(len(mask_prob)), mask_prob, alpha=0.3, color='blue')
    ax1.axhline(mask_prob.mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {mask_prob.mean():.3f}')
    ax1.set_xlabel('Frequency Bin', fontsize=12)
    ax1.set_ylabel('Masking Probability', fontsize=12)
    ax1.set_title('Frequency-wise Masking Probability', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(0, masker.n_mels - 1)
    
    # Middle: Distribution of band starting positions
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(band_starts, bins=30, color='green', alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Band Start Position', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.set_title(f'Distribution of Band Starts\n(n={len(band_starts)} bands)', 
                  fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Right: Distribution of band widths
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.hist(band_widths, bins=range(1, masker.max_band_width + 2), 
             color='orange', alpha=0.7, edgecolor='black')
    ax3.set_xlabel('Band Width (bins)', fontsize=12)
    ax3.set_ylabel('Frequency', fontsize=12)
    ax3.set_title(f'Distribution of Band Widths\n(max={masker.max_band_width})', 
                  fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.set_xticks(range(1, masker.max_band_width + 1, 2))
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n=== Summary Statistics ===")
    print(f"Total bands generated: {len(band_starts)}")
    print(f"Expected bands per sample: {masker.n_bands}")
    print(f"Actual bands per sample: {len(band_starts) / n_samples:.2f}")
    print(f"\nMasking probability:")
    print(f"  Mean: {mask_prob.mean():.3f}")
    print(f"  Std:  {mask_prob.std():.3f}")
    print(f"  Min:  {mask_prob.min():.3f}")
    print(f"  Max:  {mask_prob.max():.3f}")
    print(f"\nBand widths:")
    print(f"  Mean: {np.mean(band_widths):.2f} bins")
    print(f"  Std:  {np.std(band_widths):.2f} bins")
    print(f"  Min:  {np.min(band_widths)} bins")
    print(f"  Max:  {np.max(band_widths)} bins")

print("Function 2 (visualize_masking_statistics) defined successfully!")

### Function 3: Parameter Sensitivity Analysis

Shows how different parameter settings affect the masking behavior in a grid layout.

In [ ]:
def visualize_parameter_sensitivity(spec, param_grid):
    """
    Visualize masking behavior across different parameter settings.
    
    Args:
        spec: Test spectrogram with shape (time, freq)
        param_grid: Dictionary with parameter variations, e.g.,
                   {'n_bands': [1, 2, 3], 'max_band_width': [8, 16, 24]}
    
    Note:
        - Creates a grid showing all parameter combinations
        - Each cell shows the masked spectrogram
        - Red overlay highlights masked regions
        - Useful for understanding parameter impact
    """
    # Validate param_grid
    if not param_grid or len(param_grid) != 2:
        print("Error: param_grid must contain exactly 2 parameters")
        return
    
    # Extract parameters
    param_names = list(param_grid.keys())
    param1_name, param2_name = param_names
    param1_values = param_grid[param1_name]
    param2_values = param_grid[param2_name]
    
    print(f"Generating parameter sensitivity grid...")
    print(f"  {param1_name}: {param1_values}")
    print(f"  {param2_name}: {param2_values}")
    
    # Create grid
    n_rows = len(param1_values)
    n_cols = len(param2_values)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)
    
    # Generate masked spectrograms for each parameter combination
    for i, param1_val in enumerate(param1_values):
        for j, param2_val in enumerate(param2_values):
            print(f"  Processing {param1_name}={param1_val}, {param2_name}={param2_val}...")
            
            # Create masker with current parameters
            params = {
                'n_bands': 2,
                'mask_prob': 1.0,
                'max_band_width': 16,
                'n_mels': 128
            }
            params[param1_name] = param1_val
            params[param2_name] = param2_val
            
            masker = RandomFrequencyMasker(**params)
            masked = masker(spec)
            
            # Compute mask
            difference = np.abs(spec - masked)
            mask_binary = (difference > 0.01).astype(float)
            
            # Plot
            ax = axes[i, j]
            ax.imshow(masked.T, aspect='auto', origin='lower', cmap='viridis',
                     extent=[0, masked.shape[0], 0, masked.shape[1]])
            ax.imshow(mask_binary.T, aspect='auto', origin='lower', cmap='Reds',
                     alpha=0.3, extent=[0, masked.shape[0], 0, masked.shape[1]])
            
            # Title with parameters and masking fraction
            masked_frac = np.sum(mask_binary) / mask_binary.size
            ax.set_title(f'{param1_name}={param1_val}, {param2_name}={param2_val}\n'
                        f'Masked: {masked_frac:.1%}',
                        fontsize=11, fontweight='bold')
            ax.set_xlabel('Time', fontsize=9)
            ax.set_ylabel('Frequency', fontsize=9)
            ax.tick_params(labelsize=8)
    
    plt.suptitle(f'Parameter Sensitivity: {param1_name} vs {param2_name}',
                fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()
    
    print("\nParameter sensitivity analysis complete!")

print("Function 3 (visualize_parameter_sensitivity) defined successfully!")

## 5. Example Usage

### 5.1 Before/After Comparison

Visualize the masking effect on different spectrogram patterns.

In [ ]:
# Visualize masking on different patterns
print("Visualizing before/after comparison for different patterns...\n")

for pattern_name, spec in test_specs.items():
    print(f"\n{'='*60}")
    print(f"Pattern: {pattern_name}")
    print(f"{'='*60}")
    
    # Apply masking
    masked = masker(spec)
    
    # Visualize
    visualize_before_after(
        original=spec,
        masked=masked,
        title=f"Random Frequency Masking: {pattern_name.replace('_', ' ').title()}"
    )

### 5.2 Masking Statistics

Analyze the statistical distribution of masked positions to verify randomness.

In [ ]:
# Analyze masking statistics
print("\n" + "="*60)
print("Masking Statistics Analysis")
print("="*60 + "\n")

visualize_masking_statistics(masker, n_samples=100)

### 5.3 Parameter Sensitivity

Explore how different parameter settings affect the masking behavior.

In [ ]:
# Parameter sensitivity: n_bands vs max_band_width
print("\n" + "="*60)
print("Parameter Sensitivity: n_bands vs max_band_width")
print("="*60 + "\n")

param_grid = {
    'n_bands': [1, 2, 3],
    'max_band_width': [8, 16, 24]
}

# Use horizontal_bands pattern for clear visualization
test_spec = test_specs['horizontal_bands']
visualize_parameter_sensitivity(test_spec, param_grid)

## 6. Analysis and Insights

### Key Observations

The visualizations above reveal several important characteristics of Random Frequency Masking:

1. **Randomness**: The masking is truly random - all frequency bins have approximately equal probability of being masked (verified in statistics plot)

2. **Band Width Distribution**: Band widths are uniformly distributed from 1 to max_band_width, providing diverse augmentation

3. **Coverage**: With n_bands=2 and max_band_width=16, approximately 20-30% of frequencies are masked on average

4. **Pattern Independence**: Masking is applied uniformly regardless of spectrogram content (energy-agnostic)

5. **Parameter Impact**:
   - Increasing `n_bands` increases masking coverage linearly
   - Increasing `max_band_width` increases both coverage and variability
   - Bands may overlap, creating stronger augmentation

### Using These Insights

- **Training Robustness**: Random masking forces the model to learn features across all frequency ranges
- **Baseline Augmentation**: Provides a simple, interpretable baseline for comparing advanced masking strategies
- **Parameter Tuning**: Statistics help choose appropriate n_bands and max_band_width for your dataset
- **Validation**: Uniform distribution confirms correct implementation of random masking

### Limitations

- **Energy-Agnostic**: Doesn't consider which frequencies contain important information
- **Fixed Strategy**: Same masking probability for all frequency regions
- **No Adaptation**: Doesn't learn from model performance or data characteristics

These limitations motivate Phase 2 (Energy-Based) and Phase 3 (Activation-Guided) masking strategies.

## 7. Custom Analysis

Use the cells below for your own custom analysis and visualization experiments.

In [ ]:
# Custom analysis cell 1
# Example: Test with different mask_prob values
# for prob in [0.3, 0.5, 0.7, 1.0]:
#     masker = RandomFrequencyMasker(n_bands=2, mask_prob=prob, max_band_width=16)
#     masked = masker(test_specs['uniform'])
#     visualize_before_after(test_specs['uniform'], masked, 
#                           title=f"mask_prob={prob}")

In [ ]:
# Custom analysis cell 2
# Example: Compare masking on real vs synthetic data
# Load your own spectrogram and apply masking
# real_spec = np.load('path/to/your/spectrogram.npy')
# masked_real = masker(real_spec)
# visualize_before_after(real_spec, masked_real, title="Real Data Masking")

In [ ]:
# Custom analysis cell 3
# Example: Analyze temporal consistency
# Generate multiple masked versions of the same spectrogram
# spec = test_specs['gaussian_blob']
# for i in range(5):
#     masked = masker(spec)
#     visualize_before_after(spec, masked, title=f"Sample {i+1}")